In [4]:
import netron
netron.start('appledetection_fuji_and_mineapple.tflite')

Serving 'appledetection_fuji_and_mineapple.tflite' at http://localhost:8081


('localhost', 8081)

In [1]:
import time
import numpy as np
from ultralytics import YOLO

def benchmark_full_latency(weights_path, imgsz=320, num_warmup=10, num_runs=100, device='cpu'):
    model = YOLO(weights_path)
    dummy_img = np.random.randint(0, 255, (imgsz, imgsz, 3), dtype=np.uint8)

    # Warm-up
    for _ in range(num_warmup):
        _ = model(dummy_img, imgsz=imgsz, device=device, verbose=False)

    preprocess_times = []
    inference_times = []
    postprocess_times = []
    end_to_end_times = []

    for _ in range(num_runs):
        start_total = time.perf_counter()
        results = model(dummy_img, imgsz=imgsz, device=device, verbose=False)
        end_total = time.perf_counter()

        # Ultralytics stores per-stage timing (in ms) on the Results object's speed dict
        speed = results[0].speed  # {'preprocess': ms, 'inference': ms, 'postprocess': ms}

        preprocess_times.append(speed['preprocess'])
        inference_times.append(speed['inference'])
        postprocess_times.append(speed['postprocess'])
        end_to_end_times.append((end_total - start_total) * 1000)

    def stats(arr, label):
        arr = np.array(arr)
        print(f"{label}: mean={arr.mean():.2f} ms | std={arr.std():.2f} | min={arr.min():.2f} | max={arr.max():.2f}")
        return arr.mean()

    print(f"\n--- {weights_path} (device={device}) ---")
    mean_pre = stats(preprocess_times, "Preprocessing")
    mean_inf = stats(inference_times, "Inference")
    mean_post = stats(postprocess_times, "Postprocessing")
    mean_e2e = stats(end_to_end_times, "End-to-end (wall clock)")

    fps = 1000 / mean_e2e
    print(f"FPS (based on end-to-end): {fps:.2f}")

    return {
        'preprocess_ms': mean_pre,
        'inference_ms': mean_inf,
        'postprocess_ms': mean_post,
        'end_to_end_ms': mean_e2e,
        'fps': fps,
    }


results_minneapple = benchmark_full_latency('apple_detection_mineapple_v2.pt', device='cpu')
results_fuji = benchmark_full_latency('apple_detection_v3_fuji.pt', device='cpu')
results_merged = benchmark_full_latency('appledetection_fuji_and_mineapple.tflite', device='cpu')


--- apple_detection_mineapple_v2.pt (device=cpu) ---
Preprocessing: mean=0.88 ms | std=0.16 | min=0.77 | max=2.32
Inference: mean=52.47 ms | std=7.46 | min=46.32 | max=108.89
Postprocessing: mean=1.29 ms | std=0.24 | min=1.10 | max=3.58
End-to-end (wall clock): mean=55.02 ms | std=7.74 | min=48.57 | max=112.18
FPS (based on end-to-end): 18.18

--- apple_detection_v3_fuji.pt (device=cpu) ---
Preprocessing: mean=1.06 ms | std=0.08 | min=0.74 | max=1.26
Inference: mean=64.87 ms | std=4.54 | min=52.98 | max=77.40
Postprocessing: mean=1.58 ms | std=0.11 | min=1.20 | max=1.86
End-to-end (wall clock): mean=67.96 ms | std=4.67 | min=55.63 | max=80.79
FPS (based on end-to-end): 14.72
WARNING Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify', 'pose', 'obb' or 'semantic'.
Loading appledetection_fuji_and_mineapple.tflite for LiteRT inference...

--- appledetection_fuji_and_mineapple.tflite (device=cpu

In [7]:
import psutil
import os
import gc
from ultralytics import YOLO
import numpy as np

def measure_ram_usage(weights_path, imgsz=320, device='cpu'):
    process = psutil.Process(os.getpid())

    gc.collect()  # clean up before measuring baseline
    mem_before_load = process.memory_info().rss / (1024 ** 2)  # MB

    model = YOLO(weights_path)

    mem_after_load = process.memory_info().rss / (1024 ** 2)

    dummy_img = np.random.randint(0, 255, (imgsz, imgsz, 3), dtype=np.uint8)

    # Run a few inferences to reach steady-state memory usage
    for _ in range(10):
        _ = model(dummy_img, imgsz=imgsz, device=device, verbose=False)

    mem_after_inference = process.memory_info().rss / (1024 ** 2)

    print(f"\n--- {weights_path} ---")
    print(f"RAM before loading model: {mem_before_load:.2f} MB")
    print(f"RAM after loading model:  {mem_after_load:.2f} MB  (model footprint: {mem_after_load - mem_before_load:.2f} MB)")
    print(f"RAM after inference:      {mem_after_inference:.2f} MB  (runtime overhead: {mem_after_inference - mem_after_load:.2f} MB)")

    del model
    gc.collect()

    return {
        'model_load_mb': mem_after_load - mem_before_load,
        'inference_overhead_mb': mem_after_inference - mem_after_load,
        'total_ram_mb': mem_after_inference - mem_before_load,
    }


measure_ram_usage('appledetection_fuji_and_mineapple.tflite', device='cpu')

WARNING Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify', 'pose', 'obb' or 'semantic'.
Loading appledetection_fuji_and_mineapple.tflite for LiteRT inference...

--- appledetection_fuji_and_mineapple.tflite ---
RAM before loading model: 215.73 MB
RAM after loading model:  215.73 MB  (model footprint: 0.00 MB)
RAM after inference:      223.22 MB  (runtime overhead: 7.48 MB)


{'model_load_mb': 0.00390625,
 'inference_overhead_mb': 7.484375,
 'total_ram_mb': 7.48828125}